In [ ]:
import nescafe23 as n23
import nescafe23.beamforming as beamforming
import numpy as np
import matplotlib.pyplot as plt

import acoustic_inversion
import xarray as xr
import pygenray as pr

import cmps

In [ ]:
n23.open_peak_locations().keys()

In [ ]:
from n23_inversion_ragland.visualization import annotate_beamformer

In [ ]:
dss = n23.open_receptions()

In [ ]:
Tx_num = 0

thetas = np.linspace(-35,35,200)

upper_Rx = (
    dss[Tx_num]
    .swap_dims({"hydrophone": "nominal_depth"})
    .rename({"nominal_depth": "depth"})[:32]
)

bf, delts = beamforming.ULA_conventional_beamform_nostack(upper_Rx, thetas)

### get a ray run for reference

In [ ]:
fctd = n23.compute_filtered_fctd_grid(r_smooth=15e3, z_smooth=300)
fctd = fctd.assign_coords({'range':fctd.range + 35e3})

sound_speed = fctd['c']
bathymetry = xr.DataArray(np.linspace(5000,5000,100), coords={'range':np.linspace(0,160e3, 100)}, dims=['range'])

envi = pr.OceanEnvironment2D(sound_speed, bathymetry)

In [ ]:
source_depth = 2300
receiver_depths = [dss[0].actual_depth[:32].mean().item()]
receiver_range = 155e3
launch_angles = np.linspace(-35,35,5000)

In [ ]:
rays = pr.shoot_rays(source_depth, 0, launch_angles, receiver_range, 1000, envi)
eigs = pr.find_eigenrays(rays, receiver_depths, source_depth, 0, receiver_range, 1000, envi, max_iter=100)

### Plot beamformer output

In [ ]:
%matplotlib widget

In [ ]:
plt.close()
(20*np.log10(np.abs(bf.mean('depth'))))[::10,:].plot(x='time', vmin=85, cmap=cmps.timefront())

plt.scatter(eigs.ts[0][:,-1] - 3.15, eigs.received_angles[0],marker='o', facecolors='none', edgecolor='k', s=30)

for k in range(len(eigs.received_angles[0])):
    plt.text(eigs.ts[0][k,-1] - 3.15, eigs.received_angles[0][k], eigs.ray_id[0][k], color='k')

plt.grid()

In [ ]:
plt.close()
annotate_beamformer(bf.mean('depth')[::10,:], vmin=80)